# Exp 3 debug · AnyUp3D memory test on a T4 (T11)

Does **AnyUp3D** fit on a T4 at the Test E sizes (DINOv2-B input → propagation grid)? Only memory and time are measured; no J&F.

- **Cell 4 · one chunk:** DINOv2 on the first `chunk` frames of `MEM_CLIP`, then AnyUp3D on that chunk, for every variant × chunk size × query chunk. Peak GPU memory (whole process, so the loaded models are included) and time, or where it ran out of memory (backbone / AnyUp3D).
- **Cell 5 · whole clip:** the real Test E path for AnyUp3D: every chunk of `MEM_CLIP` upsampled and stored as the full-clip feature tensor that propagation needs (it grows while AnyUp3D runs). Propagation is not run: it does not depend on the upsampler and already fitted in run 3 at the same sizes.
- **Cell 6:** figure + CSV/JSON on Drive.

Rough time on a T4 (estimate): Cell 4 ≈ 5 min, Cell 5 ≈ 5–10 min.

In [ ]:
# ── Cell 1 · Repo + Drive + deps ────────────────────────────────────────────
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # deterministic cuBLAS (T07); must be set before anything starts CUDA
import sys, subprocess
from google.colab import drive
drive.mount("/content/drive")

REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-in-action-test"   # T11
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "scikit-image", "scipy", "-q"], check=True)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Config ─────────────────────────────────────────────────────────
MEM_CLIP     = "parkour"   # longest of the 5 Test E clips (100 frames) → largest full-clip feature tensor
CLIPS_E      = [MEM_CLIP]
OUTPUT_ROOT  = "/content/drive/MyDrive/results/T11/exp3_anyup3d_mem"
DAVIS_DEST   = "/content"
DEVICE       = "cuda"
SEED         = 0
CKPT_ANYUP3D = "/content/drive/MyDrive/results/anyup3d_step6500.pth"   # as Exp 3 (checkpoint choice S06 open)
E_CONDITIONS = ["anyup3d"]
E_VARIANTS   = [   # (name, DINOv2 input (h, w), propagation grid = AnyUp3D output (h, w)), as Test E
    ("e224_p56",          (224, 224), (56, 56)),
    ("e448_p112",         (448, 448), (112, 112)),
    ("e448x798_p112x200", (448, 798), (112, 200)),
]
MEM_CHUNKS   = [4, 8, 16]          # ↓ frames per call tried in Cell 4. AnyUp3D mask and attention ∝ chunk²
MEM_QCHUNKS  = [None, 4096, 1024]  # ↓ AnyUp3D query chunks tried in Cell 4 (None = all queries at once, as Exp 3)
E_CHUNK      = 8       # ↓ frames per call in Cell 5 (= Test E). Dependent: E_Q_CHUNK_3D
E_Q_CHUNK_3D = 1024    # ↓ AnyUp3D query chunk in Cell 5 (= Test E). Dependent: E_CHUNK
# used by the shared helpers of Cell 3 (Test E values; propagation is not run here)
E_Q_CHUNK, TAU, TOP_K, N_LAST, SHOW_H = 1024, 0.07, 7, 6, 160
E_REF = dict(input=(476, 854), tau=0.1, top_k=5, n_last=7, radius=12)

In [ ]:
# ── Cell 3 · Imports, data, helpers ─────────────────────────────────────────
import time, json, gc
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from anyup import davis, metrics, viz
from anyup.backbones import BackboneLoader
from anyup.video import iter_backbone_chunks
from anyup.upsamplers import UpsamplerLoader
from anyup.utils.seed import set_seed

set_seed(SEED)
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
ROOT = davis.download_davis(DAVIS_DEST)
missing = [c for c in CLIPS_E if c not in davis.list_clips(ROOT, "val")]
assert not missing, f"not DAVIS-2017 val clips: {missing}"
CLIPS = list(CLIPS_E)
OUT_DIR = Path(OUTPUT_ROOT) / f"n{len(CLIPS)}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(CLIPS)} clips | frames: " + ", ".join(f"{c} {len(davis.frame_paths(ROOT, c))}" for c in CLIPS) + f" | outputs → {OUT_DIR}")

E_UPS = {c: UpsamplerLoader.load(c, device=DEVICE, **(dict(ckpt_path=CKPT_ANYUP3D, q_chunk_size=E_Q_CHUNK_3D) if c == "anyup3d" else {}))
         for c in E_CONDITIONS}
from anyup.modules import attention_masking3d   # AnyUp3D caches its full masks per shape on the CPU: cleared after each variant
RES_E, PREDS_E, GRIDS_E, SKIPPED_E = {}, {}, {}, {}   # (variant, cond) → clip → eval | (T, h, w) uint8 preds; variant → grid; variant → reason


def clip_mean_jf(res):
    return (float(np.mean(res["J_M"])) + float(np.mean(res["F_M"]))) / 2


def propagate_e(feats_flat, hw, ann0, tau, top_k, n_last, radius=None, q_chunk=E_Q_CHUNK):
    """
    The davis.propagate_labels rule (Exp 3), computed in query chunks (memory ∝ q_chunk, not h·w): each query pixel keeps
    its top_k context matches (optionally only within `radius` grid cells, DINO's size_mask_neighborhood), softmax over
    those top_k with temperature tau, label = weighted mean of their soft labels.
    feats_flat (T, N, C) unit vectors on GPU; hw = (h, w), h·w = N; ann0 (H0, W0) GT of frame 0. → soft labels (T, n_ch, h, w).
    """
    T, N, C = feats_flat.shape
    h, w = hw
    n_objects = int(ann0.max())
    n_ch = n_objects + 1
    labels = torch.zeros(T, n_ch, N, device=feats_flat.device)                          # (T, n_ch, N)
    labels[0] = davis._label_maps_frame0(ann0, n_objects, h, w).to(feats_flat.device)
    if radius is not None:
        yy, xx = torch.meshgrid(torch.arange(h, device=feats_flat.device),
                                torch.arange(w, device=feats_flat.device), indexing="ij")
        pos = torch.stack([yy.flatten(), xx.flatten()], 1)                               # (N, 2) grid position
    for t in range(1, T):
        ctx = [0] + list(range(max(1, t - n_last), t))                                   # frame 0 always first
        k_feats = feats_flat[ctx].reshape(-1, C)                                         # (Nctx·N, C), frame-major
        k_labels = labels[ctx].permute(1, 0, 2).reshape(n_ch, -1)                        # (n_ch, Nctx·N)
        for q0 in range(0, N, q_chunk):
            q1 = min(N, q0 + q_chunk)
            A = feats_flat[t, q0:q1] @ k_feats.T / tau                                   # (Q, Nctx·N)
            if radius is not None:
                near = (pos[q0:q1, None, :] - pos[None, :, :]).abs().amax(-1) <= radius  # (Q, N) square window
                A = A.masked_fill(~near.repeat(1, len(ctx)), float("-inf"))
            vals, idx = A.topk(top_k, dim=-1)                                            # (Q, top_k)
            wts = F.softmax(vals, dim=-1)                                                # (Q, top_k)
            labels[t, :, q0:q1] = (k_labels[:, idx] * wts[None]).sum(-1)                 # (n_ch, Q, top_k) → (n_ch, Q)
            del A
    return labels.view(T, n_ch, h, w)


def clip_feats(bb, clip, cond, out_hw):
    """DINOv2 on every frame (E_CHUNK frames per call) → upsampled to out_hw by `cond` (None = native token grid)
    → L2-normalized, written chunk by chunk into one (T, N, C) GPU tensor (no second full-size copy)."""
    flat, t0 = None, 0
    for ch in iter_backbone_chunks(bb, davis.open_clip(ROOT, clip), chunk_size=E_CHUNK, overlap=0):
        f = ch.features if cond is None else E_UPS[cond](ch, output_size=out_hw).features   # (t, C, h, w)
        t, C, h, w = f.shape
        if flat is None:
            flat = torch.empty(len(davis.frame_paths(ROOT, clip)), h * w, C, device=f.device)   # (T, N, C)
        flat[t0:t0 + t] = F.normalize(f.reshape(t, C, h * w).permute(0, 2, 1), dim=-1)  # (t, N, C) unit vectors
        t0 += t
        del f
    assert t0 == flat.shape[0], "frame mismatch"
    return flat, (h, w)


def to_native(pred_lr, gt):
    return np.stack([davis.resize_labels(np.asarray(p, np.uint8), gt.shape[1:]) for p in pred_lr])   # (T, H0, W0), nearest


def run_e(name, input_hw, prop_hw, conds, ref=False):
    """One variant over all clips: per clip and condition DINOv2 → (upsampler) → propagation → official J&F."""
    bb = BackboneLoader.load("dinov2", device=DEVICE, input_size=input_hw,
                             guide_size=prop_hw if prop_hw else input_hw, variant="base")
    for c in conds:
        RES_E.setdefault((name, c), {}); PREDS_E.setdefault((name, c), {})
    t_start = time.perf_counter()
    failed = set()   # conditions that ran out of memory in this variant: skipped for its remaining clips, the others continue
    for ci, clip in enumerate(CLIPS):
        t0 = time.perf_counter()
        gt = davis.load_annotations(ROOT, clip)                                         # (T, 480, W) uint8
        line = []
        for c in conds:   # backbone re-run per condition: holding two full-size feature sets would double GPU memory
            if c in failed:
                continue
            flat = soft = None
            try:
                flat, hw = clip_feats(bb, clip, None if ref else c, prop_hw)
                GRIDS_E[name] = hw
                if ref:
                    soft = propagate_e(flat, hw, gt[0], E_REF["tau"], E_REF["top_k"], E_REF["n_last"], E_REF["radius"])
                    pred = torch.stack([F.interpolate(s[None], scale_factor=14, mode="bilinear", align_corners=False)[0].argmax(0)
                                        for s in soft]).cpu().numpy()                   # (T, 14h, 14w), as DINO
                else:
                    soft = propagate_e(flat, hw, gt[0], TAU, TOP_K, N_LAST)
                    pred = soft.argmax(1).cpu().numpy()                                 # (T, h, w), as Exp 3
                    if ci == 0 and c == conds[0] and name == E_VARIANTS[0][0]:          # check: same rule as davis
                        T_, N_, C_ = flat.shape
                        ref_pred = davis.propagate_labels(flat.permute(0, 2, 1).reshape(T_, C_, *hw), gt[0],
                                                          tau=TAU, top_k=TOP_K, n_last=N_LAST).numpy()
                        nd = int((ref_pred != pred).sum())
                        print(f"check propagate_e vs davis.propagate_labels on {clip}: {nd} of {pred.size} "
                              f"grid pixels differ → {'PASS' if nd <= 1e-4 * pred.size else 'WARN'}")
            except torch.cuda.OutOfMemoryError as e:
                failed.add(c)
                SKIPPED_E[f"{name}/{c}"] = (f"CUDA out of memory on {GPU_NAME} at {clip} ({str(e).splitlines()[0][:120]}); "
                                            f"reduce E_CHUNK / E_Q_CHUNK / E_Q_CHUNK_3D")
                print(f"SKIPPED {name}/{c} from {clip} on: {SKIPPED_E[f'{name}/{c}']}")
                line.append(f"{c} OOM")
                continue
            finally:
                del flat, soft
                gc.collect(); torch.cuda.empty_cache()
            RES_E[(name, c)][clip] = metrics.eval_clip_jf(gt, to_native(pred, gt))
            PREDS_E[(name, c)][clip] = pred.astype(np.uint8)
            line.append(f"{c} {clip_mean_jf(RES_E[(name, c)][clip]):.3f}")
        print(f"[{ci + 1}/{len(CLIPS)}] {clip:<16} {name}: " + "  ".join(line) + f"  {time.perf_counter() - t0:.0f}s")
    del bb
    attention_masking3d.compute_attention_mask_3d.cache_clear(); attention_masking3d.compute_attention_mask.cache_clear()
    gc.collect(); torch.cuda.empty_cache()
    print(f"{name}: grid {GRIDS_E.get(name)}, {time.perf_counter() - t_start:.0f}s\n")


def frame_and_gt(clip, t):
    """Native frame t + its GT, resized to height SHOW_H (aspect kept)."""
    img = np.array(Image.open(davis.frame_paths(ROOT, clip)[t]).convert("RGB"))
    hw = (SHOW_H, int(round(img.shape[1] * SHOW_H / img.shape[0])))
    gt = davis.load_annotation(davis.ann_paths(ROOT, clip)[t])
    return viz.resize_rgb(img, hw), davis.resize_labels(gt, hw), hw

In [ ]:
# ── Cell 4 · One chunk: variant × chunk size × query chunk ──────────────────
import psutil
LIMIT = torch.cuda.get_device_properties(0).total_memory
GB = 2 ** 30
UP3 = E_UPS["anyup3d"]
MEM = []   # one row per test


def clear():
    attention_masking3d.compute_attention_mask_3d.cache_clear(); attention_masking3d.compute_attention_mask.cache_clear()
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()


print(f"GPU {GPU_NAME}: {LIMIT / GB:.1f} GiB | models loaded: {torch.cuda.memory_allocated() / GB:.2f} GiB\n")
print(f"{'variant':<20} {'chunk':>5} {'q_chunk':>8}   {'peak GiB':>8}  {'time s':>6}  result")
for name, input_hw, prop_hw in E_VARIANTS:
    bb = BackboneLoader.load("dinov2", device=DEVICE, input_size=input_hw, guide_size=prop_hw, variant="base")
    for cs in MEM_CHUNKS:
        ch = None
        clear()
        try:
            ch = next(iter(iter_backbone_chunks(bb, davis.open_clip(ROOT, MEM_CLIP), chunk_size=cs, overlap=0)))
            bb_peak = torch.cuda.max_memory_allocated()
        except torch.cuda.OutOfMemoryError:
            for qc in MEM_QCHUNKS:
                MEM.append(dict(variant=name, chunk=cs, q_chunk=qc, peak_gib=None, time_s=None, result="OOM in DINOv2"))
                print(f"{name:<20} {cs:>5} {str(qc):>8}   {'—':>8}  {'—':>6}  OOM in DINOv2")
            continue
        lr_h, lr_w = ch.features.shape[-2:]
        mask_b = (cs * prop_hw[0] * prop_hw[1]) * (cs * lr_h * lr_w)    # AnyUp3D full bool mask, built on the CPU (≈ 2× while building)
        ram = psutil.virtual_memory().available
        if 2.5 * mask_b > ram or mask_b > 0.8 * LIMIT:                   # would crash the runtime (CPU RAM) or cannot fit the GPU
            why = f"skipped: mask {mask_b / GB:.1f} GiB (CPU RAM free {ram / GB:.1f} GiB)"
            for qc in MEM_QCHUNKS:
                MEM.append(dict(variant=name, chunk=cs, q_chunk=qc, peak_gib=None, time_s=None, result=why))
                print(f"{name:<20} {cs:>5} {str(qc):>8}   {'—':>8}  {'—':>6}  {why}")
            del ch
            continue
        for qc in MEM_QCHUNKS:
            UP3.q_chunk_size = qc
            out = None
            clear()
            try:
                torch.cuda.synchronize(); t0 = time.perf_counter()
                out = UP3(ch, output_size=prop_hw).features                            # (cs, 768, H, W)
                torch.cuda.synchronize()
                dt, peak = time.perf_counter() - t0, max(torch.cuda.max_memory_allocated(), bb_peak)
                res = "OK"
            except torch.cuda.OutOfMemoryError as e:
                dt, peak, res = None, None, "OOM in AnyUp3D"
            del out
            MEM.append(dict(variant=name, chunk=cs, q_chunk=qc, peak_gib=None if peak is None else peak / GB, time_s=dt, result=res))
            print(f"{name:<20} {cs:>5} {str(qc):>8}   {'—' if peak is None else f'{peak / GB:8.2f}':>8}  "
                  f"{'—' if dt is None else f'{dt:6.1f}':>6}  {res}")
        del ch
    del bb
    clear()
UP3.q_chunk_size = E_Q_CHUNK_3D
print("\npeak = highest GPU memory of the process during the call (models + chunk + AnyUp3D), incl. the DINOv2 step")

In [ ]:
# ── Cell 5 · Whole clip, the real Test E path (E_CHUNK, E_Q_CHUNK_3D) ────────
T_CLIP = len(davis.frame_paths(ROOT, MEM_CLIP))
CLIP_MEM = []
print(f"{MEM_CLIP}: {T_CLIP} frames | E_CHUNK {E_CHUNK} | E_Q_CHUNK_3D {E_Q_CHUNK_3D}\n")
for name, input_hw, prop_hw in E_VARIANTS:
    bb = BackboneLoader.load("dinov2", device=DEVICE, input_size=input_hw, guide_size=prop_hw, variant="base")
    flat = None
    clear()
    try:
        t0 = time.perf_counter()
        flat, hw = clip_feats(bb, MEM_CLIP, "anyup3d", prop_hw)                          # (T, N, 768) GPU, as Test E
        torch.cuda.synchronize()
        dt, peak, res = time.perf_counter() - t0, torch.cuda.max_memory_allocated(), "OK"
        store = flat.numel() * 4
    except torch.cuda.OutOfMemoryError as e:
        dt, peak, res, store = None, None, "OOM", T_CLIP * prop_hw[0] * prop_hw[1] * 768 * 4
    del flat
    CLIP_MEM.append(dict(variant=name, peak_gib=None if peak is None else peak / GB, store_gib=store / GB, time_s=dt, result=res))
    print(f"{name:<20} feature store {store / GB:5.2f} GiB | peak {'—' if peak is None else f'{peak / GB:.2f}'} GiB of "
          f"{LIMIT / GB:.1f} | {'—' if dt is None else f'{dt:.0f}s'} | {res}")
    del bb
    clear()
print("\nPropagation (not run) adds about E_Q_CHUNK × (1+N_LAST)·h·w floats ×≈3 on top of the feature store "
      "(112×200: ≈ 2 GB); it fitted in run 3 at the same sizes.")

In [ ]:
# ── Cell 6 · Figure + save ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(E_VARIANTS) + 1, figsize=(4.3 * (len(E_VARIANTS) + 1), 4.2))
for ax, (name, _, prop_hw) in zip(axes, E_VARIANTS):
    M = np.full((len(MEM_CHUNKS), len(MEM_QCHUNKS)), np.nan)
    for r in MEM:
        if r["variant"] == name and r["peak_gib"] is not None:
            M[MEM_CHUNKS.index(r["chunk"]), MEM_QCHUNKS.index(r["q_chunk"])] = r["peak_gib"]
    ax.imshow(M, cmap="RdYlGn_r", vmin=0, vmax=LIMIT / GB)
    for r in MEM:
        if r["variant"] == name:
            i, j = MEM_CHUNKS.index(r["chunk"]), MEM_QCHUNKS.index(r["q_chunk"])
            txt = (f"{r['peak_gib']:.1f} GB\n{r['time_s']:.1f}s" if r["result"] == "OK" else
                   "mask too big\n(not run)" if r["result"].startswith("skipped") else r["result"].replace(" in ", "\n"))
            ax.text(j, i, txt, ha="center", va="center", fontsize=8, color="k" if r["result"] == "OK" else "darkred")
    ax.set_xticks(range(len(MEM_QCHUNKS))); ax.set_xticklabels([str(q) for q in MEM_QCHUNKS]); ax.set_xlabel("AnyUp3D q_chunk")
    ax.set_yticks(range(len(MEM_CHUNKS))); ax.set_yticklabels(MEM_CHUNKS); ax.set_ylabel("frames per call")
    ax.set_title(f"{name}\n→ {prop_hw[0]}×{prop_hw[1]}, one chunk", fontsize=9)
ax = axes[-1]
x = np.arange(len(CLIP_MEM))
ax.bar(x, [r["peak_gib"] or 0 for r in CLIP_MEM], color=["tab:green" if r["result"] == "OK" else "tab:red" for r in CLIP_MEM])
ax.bar(x, [r["store_gib"] for r in CLIP_MEM], 0.4, color="none", edgecolor="k", hatch="//", label="feature store")
for i, r in enumerate(CLIP_MEM):
    ax.text(i, (r["peak_gib"] or 0) + 0.2, f"{r['peak_gib']:.1f}" if r["result"] == "OK" else "OOM", ha="center", fontsize=8)
ax.axhline(LIMIT / GB, color="r", ls="--", lw=1, label=f"{GPU_NAME} ({LIMIT / GB:.1f} GiB)")
ax.set_xticks(x); ax.set_xticklabels([r["variant"] for r in CLIP_MEM], fontsize=7, rotation=15)
ax.set_ylabel("peak GiB"); ax.legend(fontsize=7); ax.grid(axis="y", alpha=0.3)
ax.set_title(f"whole clip ({MEM_CLIP}, {T_CLIP} frames)\nchunk {E_CHUNK}, q_chunk {E_Q_CHUNK_3D}", fontsize=9)
fig.suptitle(f"AnyUp3D memory on {GPU_NAME} (green = fits, red = near the limit; white = out of memory / not run)", fontsize=11)
plt.tight_layout(); plt.savefig(OUT_DIR / "anyup3d_memory.png", dpi=130, bbox_inches="tight"); plt.show()
config = dict(MEM_CLIP=MEM_CLIP, E_VARIANTS=E_VARIANTS, MEM_CHUNKS=MEM_CHUNKS, MEM_QCHUNKS=MEM_QCHUNKS, E_CHUNK=E_CHUNK,
              E_Q_CHUNK_3D=E_Q_CHUNK_3D, CKPT_ANYUP3D=CKPT_ANYUP3D, gpu=GPU_NAME, repo_branch=REPO_BRANCH)
csv_p, json_p = metrics.save_results(OUT_DIR, MEM, dict(whole_clip=CLIP_MEM, limit_gib=LIMIT / GB), config, name="anyup3d_memory")
print("saved:", csv_p, json_p, OUT_DIR / "anyup3d_memory.png")